# Country Flags Web Scraping

This notebook collects country information from **Flagpedia** and creates a reusable dataset containing general country data and direct links to SVG flag images.

The scraping process extracts the following information for each country:

- country name;
- area in km²;
- population;
- country abbreviation;
- SVG flag URL.

The final dataset is exported as `country-flags.csv` so it can be reused without running the scraping process again.

## Notebook structure

1. Libraries and project setup
2. HTTP request headers
3. Requesting and parsing the webpage
4. Extracting Country Information
5. Standardizing Country Names

## 1. Libraries and Project Setup

The notebook uses three main libraries:

- **requests** to send the HTTP request to Flagpedia;
- **BeautifulSoup** to parse and navigate the returned HTML;
- **pandas** to organize the extracted records into a structured DataFrame and export them as a CSV file.

In [9]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

## 2. HTTP Request Headers

A browser-like `User-Agent` is included in the request headers.

This makes the request resemble one sent by a regular web browser and helps reduce the chance of the website rejecting the request.

In [10]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

## 3. Requesting and Parsing the Webpage

The data is collected from the Flagpedia index page, which contains the complete list of country flags.

After sending the request, the returned HTML is parsed with BeautifulSoup using the `lxml` parser. The HTTP status code is displayed so the request can be checked before continuing with the extraction.

In [11]:
url = f'https://flagpedia.net/index'
response = requests.get(url, headers=headers)
response.status_code
soup = BeautifulSoup(response.content, "lxml")

## 4. Extracting Country Information

For every country in `flag_list`, the scraper extracts:

- `country_abr` — country abbreviation obtained from the flag image path;
- `country_name` — country name;
- `country_url` — direct URL to the SVG version of the country's flag;
- `country_area` — country area in km²;
- `country_population` — population value provided by Flagpedia.

The area and population values are converted to `float`, while the remaining fields are stored as strings.

In [12]:
# Locating the section containing all country flags
all_flags = soup.find_all('ul',{'class':'flag-grid'})
flag_list = all_flags[0].find_all('li')

flags = []

# Extracting information for each country
for row in flag_list:
    temp = []

    # Extract the country name, area, and population.
    country_name = row.find('span').string
    country_area = row.find('a').get('data-area')
    country_population = row.find('a').get('data-population')

    # Extract the country abbreviation from the flag image URL.
    country_abr = row.find('img').get('src')
    country_abr = country_abr.split('/')[-1]
    country_abr = country_abr.split('.')[0]

    # Create the direct URL to the SVG version of the flag.
    country_url = f'https://flagcdn.com/{country_abr}.svg'

    # Store the extracted values in a dictionary.
    temp = {
        'country_abr': country_abr, 
        'country_name': country_name,
        'country_url': country_url,
        'country_area': float(country_area),
        'country_population': int(country_population)

    }

    flags.append(temp)

# Creating and exporting the dataset
df_flags = pd.DataFrame(flags)
display(df_flags)

,country_abr,country_name,country_url,country_area,country_population
0,af,Afghanistan,https://flagcdn.com/af.svg,652230.0,32225560
1,ax,Åland Islands,https://flagcdn.com/ax.svg,1580.0,29789
2,al,Albania,https://flagcdn.com/al.svg,28748.0,2862427
3,dz,Algeria,https://flagcdn.com/dz.svg,2381740.0,43000000
4,as,American Samoa,https://flagcdn.com/as.svg,199.0,56700
...,...,...,...,...,...
249,wf,Wallis and Futuna,https://flagcdn.com/wf.svg,142.0,11700
250,eh,Western Sahara,https://flagcdn.com/eh.svg,266000.0,582463
251,ye,Yemen,https://flagcdn.com/ye.svg,527968.0,29161922
252,zm,Zambia,https://flagcdn.com/zm.svg,752612.0,17381168


## 5. Standardizing Country Names

Some country names returned by Transfermarkt use a different naming convention from the country names available in the flags dataset.

To ensure that both datasets can be correctly matched, the country names were standardized before the merge. The values that differed between the two sources were identified and replaced with their corresponding names from the flags dataset.

In [13]:
# Map Transfermarkt country names to the corresponding names used in the flags dataset
country_name_mapping = {
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Republic of the Congo": "Congo",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Curaçao": "Curacao",
    "Czechia": "Czech Republic",
    "South Korea": "Korea, South",
    "Saint Kitts and Nevis": "St. Kitts & Nevis",
    "Gambia": "The Gambia",
    "Turkey": "Türkiye"
}

# Create a copy to preserve the original DataFrame
new_df = df_flags.copy()

# Standardize country names
new_df['country_name'] = new_df['country_name'].replace(country_name_mapping)

new_df.to_csv('../data-csv/country-flags.csv', index=False)
display(new_df)

,country_abr,country_name,country_url,country_area,country_population
0,af,Afghanistan,https://flagcdn.com/af.svg,652230.0,32225560
1,ax,Åland Islands,https://flagcdn.com/ax.svg,1580.0,29789
2,al,Albania,https://flagcdn.com/al.svg,28748.0,2862427
3,dz,Algeria,https://flagcdn.com/dz.svg,2381740.0,43000000
4,as,American Samoa,https://flagcdn.com/as.svg,199.0,56700
...,...,...,...,...,...
249,wf,Wallis and Futuna,https://flagcdn.com/wf.svg,142.0,11700
250,eh,Western Sahara,https://flagcdn.com/eh.svg,266000.0,582463
251,ye,Yemen,https://flagcdn.com/ye.svg,527968.0,29161922
252,zm,Zambia,https://flagcdn.com/zm.svg,752612.0,17381168


### Final Notes

The country name standardization ensures consistency between the data collected from Transfermarkt and the flags dataset. Only countries with different naming conventions between the two sources were modified, while all other values remained unchanged.

This step is necessary to prevent missing matches when combining the datasets and allows each country to be correctly associated with its corresponding flag information. If new naming differences are identified in future data, they can be easily added to the `country_name_mapping` dictionary.